In [2]:
import gurobipy as gp
import numpy as np
from gurobipy import GRB
from gurobipy import *

In [3]:
# parameters
Operations = [1,2,3,4]
J= len(Operations)
Ressources = [1,2,3,4]
M= len(Ressources)
T= 20

predecessors = [[],
                [0],
                [0],
                [1, 2]]

successors = [[] for _ in range(J)]
for j in range(J):
    for h in predecessors[j]:
        successors[h].append(j)

duration = [2, 3, 2, 4]

demand_capacity =  [[1, 1, 2, 1],
                    [2, 1, 1, 2],
                    [1, 2, 1, 1],
                    [2, 2, 2, 2]]

FEZ= [0,0,0,0]
SEZ= [15,15,15,15]

capacity= [5,5,5,5]

In [4]:
m = gp.Model("RCPSP")

Set parameter Username
Set parameter LicenseID to value 2725807
Academic license - for non-commercial use only - expires 2026-10-21


In [5]:
# decision variables
S = m.addVars(J, T, vtype= GRB.BINARY)
C = m.addVar(lb= 0, vtype= GRB.CONTINUOUS)

In [6]:
m.setObjective(C, GRB.MINIMIZE)

In [7]:
for j in range(J):
    m.addConstr(C >= sum((t + duration[j]) * S[j,t] for t in range(FEZ[j], SEZ[j]+1)))

In [8]:
for j in range(J):
    m.addConstr(quicksum(S[j, t] for t in range(T)) == 1)

In [9]:
for j in range(J):
    m.addConstr((quicksum(S[j, t] for t in range(FEZ[j], SEZ[j]+1)) == 1))

In [10]:
# disjunction constraint
for j in range(J):
        for h in predecessors[j]:
                m.addConstr(quicksum(t * S[h, t] for t in range(FEZ[h], SEZ[h]+1)) <= quicksum((t - duration[j]) * S[j, t] for t in range(FEZ[j], SEZ[j]+1)))


In [11]:
for r in range(M):
    for t in range(T):
        m.addConstr((quicksum(demand_capacity[j][r] * quicksum(S[j,q] for q in range(t, min(t+duration[j], T))) for j in range(J)) <= capacity[r]))

In [12]:
# Solve
m.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 96 rows, 81 columns and 1168 nonzeros
Model fingerprint: 0x455bccf9
Variable types: 1 continuous, 80 integer (80 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]
Found heuristic solution: objective 15.0000000
Presolve removed 96 rows and 81 columns
Presolve time: 0.03s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.10 seconds (0.00 work units)
Thread count was 1 (of 12 available processors)

Solution count 2: 11 15 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.100000000000e+01, best bound 1.100000000000e+01, gap 0.0000%


In [13]:
#m.computeIIS()
#m.write("rcpsp.ilp")

In [14]:
print("Objective value: ", m.objVal , " found after ", m.Runtime, " seconds. Relative gap is: ", m.MIPGap)

Objective value:  11.0  found after  0.12100005149841309  seconds. Relative gap is:  0.0
